In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py

import pandas as pd
from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
from pymongo import MongoClient
import pymysql
from datetime import datetime,timedelta
import matplotlib.pyplot as plt
import psycopg2




db = client.get_database('telematics_db')


db2 = client2.get_database('telematics_db')


gettime=lambda x: pd.Timestamp(x).timestamp()
getdate=lambda x: pd.datetime.fromtimestamp(x).date()
getDay=lambda x: pd.datetime.fromtimestamp(x).date()
dttoday=int(tm.time()-tm.time()%86400-19800)

In [ ]:
a=pd.read_csv("Airtel_voda_installer_counts.csv")

In [ ]:
final=[]
for i in a.installer_name.unique():
    airtel=a[a['network']=='AIRTEL'][a['installer_name']==i]['Distinct values of sim_id'].sum()
    voda=a[a['network']=='VODAFONE'][a['installer_name']==i]['Distinct values of sim_id'].sum()
    final.append([i,airtel,voda,voda/(airtel+voda)*100])
    

In [ ]:
yo=pd.DataFrame(final,columns=['installer_name_location','airtel_count','voda_count','voda_percent'])

In [ ]:
yo=yo[yo['airtel_count']>10][yo['voda_percent']>25]

In [ ]:
yo.sort_values(['voda_percent','voda_count'],ascending=False).to_csv("high_voda_percent_installer_locations.csv") 


In [ ]:
installers=pd.read_sql("select name as installer_name_location,phone_no,code from fleet_lake.users where name in {}".format(tuple(a.installer_name.dropna().unique())),con)  


In [ ]:
ji=pd.read_csv("high_voda_percent_installer_locations.csv")

In [ ]:
del ji['Unnamed: 0']

In [ ]:
ji=pd.merge(ji,installers,how='left',on='installer_name_location')

In [ ]:
ji.to_csv("high_voda_percent_installer_locations.csv")

In [ ]:
ji[ji['airtel_count']>30][ji['voda_percent']>80]


In [ ]:
installers_vehicles_to_test=ji[ji['airtel_count']>30][ji['voda_percent']>80].code

In [ ]:
vehicles=pd.read_sql("select vehicle_id,network,actual_live_time from analytics.vehicle_details where installer_code in {} ".format(tuple(installers_vehicles_to_test)),con) 


In [ ]:
vehicles

In [ ]:
vehicles=vehicles[vehicles.network.isin(['VODAFONE','AIRTEL'])][vehicles['actual_live_time']>'2020-01-01']


In [ ]:
iterable=[]
for i,row in vehicles.iterrows():
    iterable.append([int(row['vehicle_id']),int(gettime(row['actual_live_time'])),int(gettime(row['actual_live_time'])+86400*30)])
    
    

In [ ]:
def no_info_analysis(l):
    print(l)
    vehicle_id,fromTime,toTime=l
    no_info=pd.DataFrame(list(db.deviceHistory.find({'vehicleId':vehicle_id,'fromTime':{'$gte':fromTime},'toTime':{'$lte':toTime}})))
    number_of_times=len(no_info)

    if number_of_times>0:
        no_info['total_time']=no_info['toTime']-no_info['fromTime']
        total_noinfo=no_info['total_time'].sum()
        avg_time=total_noinfo/number_of_times
    else:
        avg_time=0
        total_noinfo=0
    
    return [vehicle_id,number_of_times,avg_time,total_noinfo]


In [ ]:
import multiprocessing
pool=multiprocessing.Pool()
results=pool.map(no_info_analysis,iterable)
pool.close()
pool.join()


In [ ]:
final=pd.DataFrame(results,columns=['vehicle_id','number_of_times','avg_time','total_noinfo'])

In [ ]:
final=pd.merge(final,vehicles,on='vehicle_id',how='left')

In [ ]:
final.total_noinfo=final.total_noinfo/3600


In [ ]:
final.avg_time=final.avg_time/3600

In [ ]:
plt.hist(final[final['network']=='VODAFONE']['number_of_times'],bins=range(0,100,1))
plt.hist(final[final['network']=='AIRTEL']['number_of_times'],bins=range(0,100,1))

In [ ]:
plt.figure(figsize=(20,20))
plt.hist(final[final['network']=='VODAFONE']['total_noinfo'],bins=range(0,300,1))
plt.hist(final[final['network']=='AIRTEL']['total_noinfo'],bins=range(0,300,1))


In [ ]:
print("total hours of noinfo,in the first month after installation")
final.groupby('network').total_noinfo.describe()


In [ ]:
print("number of times device went in no info,in the first month after installation")
final.groupby('network').number_of_times.describe()

In [ ]:
1419+399.0

In [ ]:
1419.0/1818.0